# Uniform Robust Pricing Report

This notebook uses the packaged `mot_pricing` library rather than duplicating solver code inside notebook cells.

Problem setup:
- `S1 ~ Uniform[1, 3]`
- `S2 ~ Uniform[0, 4]`
- payoff `|S1 - S2|`
- exact martingale constraint `E[S2 | S1] = S1`


In [ ]:
from mot_pricing import run_uniform_abs_spread_experiment

experiment = run_uniform_abs_spread_experiment(
    n=50,
    eps_values=(1.0, 0.3, 0.1, 0.03, 0.01),
    sinkhorn_iterations=600,
    sinkhorn_tolerance=1e-8,
)

experiment.exact_upper.value, experiment.exact_lower.value


In [ ]:
print("Exact MOT upper:", f"{experiment.exact_upper.value:.6f}")
print("Exact MOT lower:", f"{experiment.exact_lower.value:.6f}")
print("Comonotone minimum:", f"{experiment.unrestricted_benchmarks.unrestricted_min_comonotone:.6f}")
print("Independent:", f"{experiment.unrestricted_benchmarks.independent:.6f}")
print("Countermonotone maximum:", f"{experiment.unrestricted_benchmarks.unrestricted_max_countermonotone:.6f}")
for eps, result in sorted(experiment.regularized_results.items()):
    print(
        f"eps={eps:.4f}: expected={result.expected_payoff:.6f}, "
        f"regularized={result.regularized_primal:.6f}, dual_gap={result.dual_gap:.2e}"
    )


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

im = axes[0].imshow(
    experiment.exact_upper.plan.T,
    origin="lower",
    aspect="auto",
    extent=[
        experiment.x_atoms[0],
        experiment.x_atoms[-1],
        experiment.y_atoms[0],
        experiment.y_atoms[-1],
    ],
    cmap="YlOrRd",
)
axes[0].plot([1, 3], [1, 3], "k--", linewidth=1.0)
axes[0].set_title("Exact MOT upper plan")
axes[0].set_xlabel("s1")
axes[0].set_ylabel("s2")
fig.colorbar(im, ax=axes[0])

axes[1].bar(
    ["Lower", "Comonotone", "Independent", "Upper", "Countermonotone"],
    [
        experiment.exact_lower.value,
        experiment.unrestricted_benchmarks.unrestricted_min_comonotone,
        experiment.unrestricted_benchmarks.independent,
        experiment.exact_upper.value,
        experiment.unrestricted_benchmarks.unrestricted_max_countermonotone,
    ],
)
axes[1].set_title("Coupling benchmarks")
axes[1].set_ylabel("Expected payoff")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()


In [ ]:
eps_values = sorted(experiment.regularized_results)
expected = [experiment.regularized_results[eps].expected_payoff for eps in eps_values]
regularized = [experiment.regularized_results[eps].regularized_primal for eps in eps_values]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].semilogx(eps_values, expected, "o-", linewidth=2)
axes[0].axhline(experiment.exact_upper.value, linestyle="--", color="red")
axes[0].set_title("Expected payoff versus eps")
axes[0].set_xlabel("eps")
axes[0].set_ylabel("E[|S1-S2|]")
axes[0].grid(alpha=0.3)

axes[1].semilogx(eps_values, regularized, "o-", linewidth=2, color="green")
axes[1].set_title("Regularized objective versus eps")
axes[1].set_xlabel("eps")
axes[1].set_ylabel("Regularized objective")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()
